# Exp7.3.4 — Linear-pretrained LIF fine-tuning

Analysis-only notebook. It reads finalized artifacts and compares direct transfer, pretrained LIF fine-tuning, and random-init LIF training at the method level.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ART = ROOT / 'notebooks' / 'artifacts' / 'experiment_7_3_4_linear_pretrained_lif_finetuning' / 'linear_pretrained_lif_finetuning_v1'
manifest = json.loads((ART / 'manifest.json').read_text())
method_runs = pd.read_csv(ART / 'method_runs.csv')
method_summary = pd.read_csv(ART / 'method_summary.csv')
contrast_summary = pd.read_csv(ART / 'contrast_summary.csv')
history_summary = pd.read_csv(ART / 'history_summary.csv')
checkpoint_diagnostics = pd.read_csv(ART / 'checkpoint_diagnostics.csv')
source_checks = pd.read_csv(ART / 'source_reproduction_checks.csv')
manifest

## Primary method comparison

In [ ]:
display(method_summary[['method', 'checkpoint_role', 'test_lif_ba_mean', 'test_lif_ba_std', 'test_linear_ba_mean', 'test_linear_ba_std']])
plot_df = method_summary.set_index('method')
ax = plot_df['test_lif_ba_mean'].plot(kind='bar', yerr=plot_df['test_lif_ba_std'], capsize=4)
ax.set_ylabel('Test balanced accuracy')
ax.set_xlabel('Method')
ax.set_title('Exp7.3.4 — final LIF test BA')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Paired contrasts

In [ ]:
display(contrast_summary.sort_values('contrast'))

## Fine-tuning trajectory

Solid interpretation target: whether LIF BA improves while the same current W remains linearly discriminative, or whether LIF optimization trades away the Linear geometry.

In [ ]:
for init_source in ['a2', 'b6', 'random']:
    d = history_summary[history_summary.init_source == init_source].sort_values('epoch')
    fig, ax = plt.subplots()
    ax.plot(d['epoch'], d['val_lif_ba_mean'], label='Val LIF BA')
    ax.plot(d['epoch'], d['val_linear_ba_mean'], label='Val Linear-bypass BA')
    ax.set_title(f'{init_source}: validation trajectory')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Balanced accuracy')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Weight drift

In [ ]:
for init_source in ['a2', 'b6']:
    d = history_summary[history_summary.init_source == init_source].sort_values('epoch')
    fig, ax = plt.subplots()
    ax.plot(d['epoch'], d['weight_relative_frobenius_drift_mean'], label='Relative Frobenius drift')
    ax.plot(d['epoch'], d['weight_mean_class_cosine_mean'], label='Mean class cosine')
    ax.set_title(f'{init_source}: W drift from Linear initialization')
    ax.set_xlabel('Epoch')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Output spike dynamics

In [ ]:
for init_source in ['a2', 'b6', 'random']:
    d = history_summary[history_summary.init_source == init_source].sort_values('epoch')
    fig, ax = plt.subplots()
    ax.plot(d['epoch'], d['val_mean_total_output_spikes_per_sample_mean'], label='Mean output spikes/sample')
    ax.plot(d['epoch'], d['val_silent_sample_fraction_mean'], label='Silent-sample fraction')
    ax.set_title(f'{init_source}: output spike dynamics')
    ax.set_xlabel('Epoch')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Audit tables

In [ ]:
display(checkpoint_diagnostics.sort_values(['init_source', 'seed', 'state']))
display(source_checks.sort_values(['init_source', 'seed']))